# 🤖 EchoNest Business Agent — EECE 503P/798S Assignment C3

**Milo**, EchoNest's Memory Concierge: persona + instructions, intent router, tool calls, short-term memory, Gradio UI.

**Sections:** 1 Setup · 2 Business Context · 3 🎭 Persona · 4 📏 Instructions · 5 System Prompt · 6 Tools + Schemas · 7 Router · 8 Agent Loop · 9 Orchestrator · 10 Reflection · 11 Tests · 12 Gradio

## 1. Setup

### 1.1 Imports and configuration
All paths and the model name live in one place so the same code works in this notebook, `app.py`, and Hugging Face Spaces.

In [ ]:
import os
import sys
from pathlib import Path

import openai, gradio, pypdf, dotenv

ME_DIR = Path("me")
LOGS_DIR = Path("logs")
LOGS_DIR.mkdir(exist_ok=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.5-flash-lite"   # confirm the exact current name in AI Studio

print("Python :", sys.version.split()[0], "@", sys.executable)
print("openai :", openai.__version__)
print("gradio :", gradio.__version__)
print("pypdf  :", pypdf.__version__)
print("cwd    :", os.getcwd())

### 1.2 Load the API key
Works in both places: **Colab Secrets** when running in Colab, otherwise the local **`.env`** file (git-ignored).
The key is never printed in full.

In [ ]:
try:
    from google.colab import userdata          # only exists in Colab
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import find_dotenv, load_dotenv   # local Jupyter / app.py
    load_dotenv(find_dotenv(usecwd=True))        # search from the working directory

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "API key not found — add GEMINI_API_KEY to .env (or Colab Secrets)"
print("✅ Key loaded:", GEMINI_API_KEY[:4] + "...")

## 2. Business Context

Milo may only answer from two files in `me/`: `business_summary.txt` and `about_business.pdf`.
The PDF is turned into a string with `pypdf` (`page.extract_text()` for each page), the pages are joined with blank
lines, and extra whitespace is cleaned up. Both files are small (a few thousand characters), so they go straight into
the system prompt in Section 5. A project this size doesn't need retrieval (RAG).

In [ ]:
import re
from pypdf import PdfReader

BUSINESS_NAME = "EchoNest"


def clean_text(text):
    """Normalize line endings, collapse runs of spaces, trim each line, and allow at most one blank line."""
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_business_context(me_dir=ME_DIR):
    """Return (summary_text, pdf_text) loaded from me/business_summary.txt and me/about_business.pdf."""
    me_dir = Path(me_dir)
    summary_path = me_dir / "business_summary.txt"
    pdf_path = me_dir / "about_business.pdf"
    for path in (summary_path, pdf_path):
        if not path.exists():
            raise FileNotFoundError(f"{path} not found (cwd = {os.getcwd()}). Run from the repo root.")

    summary_text = clean_text(summary_path.read_text(encoding="utf-8"))
    reader = PdfReader(pdf_path)
    pdf_text = clean_text("\n\n".join((page.extract_text() or "") for page in reader.pages))
    return summary_text, pdf_text

In [ ]:
summary_text, pdf_text = load_business_context()

for name, text in [("business_summary.txt", summary_text), ("about_business.pdf", pdf_text)]:
    print(f"📄 {name}: {len(text):,} chars (~{len(text) // 4:,} tokens)")
    print(text[:300], "...\n")

### 2.1 Context checks (no API calls)
These use the Stage 1 test plan in `tests/knowledge_checks.json`. Whitespace is collapsed before matching, because
the PDF can break a phrase across two lines.

In [ ]:
import json

checks = json.loads(Path("tests/knowledge_checks.json").read_text(encoding="utf-8"))


def flat(text):
    return re.sub(r"\s+", " ", text)


pdf_flat, summary_flat = flat(pdf_text), flat(summary_text)

assert len(pdf_text) > 500, "PDF text too short. Is the PDF a scanned image?"
assert len(summary_text) > 200, "Summary too short"
assert BUSINESS_NAME in pdf_text and BUSINESS_NAME in summary_text, "Business name missing"
for item in checks["pdf_only"]:
    assert item["phrase"] in pdf_flat, f"PDF-only fact missing: {item['phrase']!r}"
for item in checks["summary_only"]:
    assert item["phrase"] in summary_flat, f"Summary-only fact missing: {item['phrase']!r}"
for fact in checks["shared_facts"]:
    assert fact in pdf_flat and fact in summary_flat, f"Shared fact missing: {fact!r}"

total = len(pdf_text) + len(summary_text)
print(f"✅ Context OK: {len(checks['pdf_only'])} PDF-only, {len(checks['summary_only'])} summary-only and "
      f"{len(checks['shared_facts'])} shared facts found ({total:,} chars total, ~{total // 4:,} tokens)")